# Apache Access Logs with Spark RDDs

This notebook uses Apache Spark 3.5.3 RDDs to parse Apache access logs stored in HDFS. It does not use DataFrames.

You will download a sample log, upload it to your HDFS directory, parse each line with a regular expression, analyze the records, and save CSV-formatted text to HDFS.

## 1. Start HDFS and Spark

Run these commands in a WSL terminal before starting the notebook:

```bash
start-dfs.sh
/opt/spark/sbin/start-master.sh
/opt/spark/sbin/start-worker.sh "spark://$(hostname):7077"
jps
```

This lesson uses the Spark standalone master, not YARN.

## 2. Download the sample Apache log

The source file is from Elastic's public examples repository:

<https://raw.githubusercontent.com/elastic/examples/refs/heads/master/Common%20Data%20Formats/apache_logs/apache_logs>

Download it to `/tmp`. The file is temporary local input; the lesson's durable copy will be stored in HDFS.

In [ ]:
%%bash
SOURCE_URL='https://raw.githubusercontent.com/elastic/examples/refs/heads/master/Common%20Data%20Formats/apache_logs/apache_logs'
LOCAL_FILE='/tmp/apache_logs'

wget -O "$LOCAL_FILE" "$SOURCE_URL"
ls -lh "$LOCAL_FILE"
wc -l "$LOCAL_FILE"
head -3 "$LOCAL_FILE"

## 3. Upload the log to HDFS

The destination uses `$USER`, so every student works in a separate HDFS directory. `-put -f` safely replaces this input file when the cell is repeated.

In [ ]:
%%bash
HDFS_LAB="/user/$USER/apache-logs"

hdfs dfs -mkdir -p "$HDFS_LAB"
hdfs dfs -put -f /tmp/apache_logs "$HDFS_LAB/apache_logs"
hdfs dfs -ls -h "$HDFS_LAB"
hdfs dfs -head "$HDFS_LAB/apache_logs"

## 4. Connect to the Spark standalone master

Spark uses `HADOOP_CONF_DIR` to find the HDFS NameNode.

In [ ]:
import os
import socket

from pyspark.sql import SparkSession

master_url = os.environ.get(
    "SPARK_MASTER",
    f"spark://{socket.gethostname()}:7077",
)

spark = (
    SparkSession.builder
    .appName("D282-Apache-Logs-RDD")
    .master(master_url)
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("WARN")

print("Spark version :", spark.version)
print("Spark master  :", sc.master)
print("Application ID:", sc.applicationId)
print("Spark UI      :", sc.uiWebUrl)

## 5. Read the log from HDFS

`textFile` returns an RDD containing one log line per record. Reading is lazy until an action is called.

In [ ]:
hdfs_user = os.environ["USER"]
hdfs_input = f"hdfs:///user/{hdfs_user}/apache-logs/apache_logs"
hdfs_csv_output = f"hdfs:///user/{hdfs_user}/apache-logs/output/csv"

raw_logs_rdd = sc.textFile(hdfs_input)

print("Input path      :", hdfs_input)
print("Input partitions:", raw_logs_rdd.getNumPartitions())

In [ ]:
raw_logs_rdd.take(3)

## 6. Understand the log format

A typical combined Apache access-log line contains:

```text
client_ip identity user [timestamp zone] "method path protocol" status bytes "referrer" "user_agent"
```

A dash means that the source did not provide a value. Paths and user-agent strings may contain spaces or special characters, so splitting only on spaces is unreliable.

## 7. Create the regular expression

Named groups make the parsed fields clear. The final referrer and user-agent fields are optional so the parser can also accept the common log format.

In [ ]:
import re

APACHE_LOG_PATTERN = re.compile(
    r'^(?P<client_ip>\S+) '
    r'(?P<identity>\S+) '
    r'(?P<user>\S+) '
    r'\[(?P<timestamp>[^]]+)\] '
    r'"(?P<method>\S+) (?P<path>.*?) (?P<protocol>\S+)" '
    r'(?P<status>\d{3}) '
    r'(?P<bytes>\S+)'
    r'(?: "(?P<referrer>[^"]*)" "(?P<user_agent>[^"]*)")?$'
)

## 8. Define the parsing function

The function returns a dictionary for a valid line and `None` for a malformed line. Status and byte count are converted to integers. A dash in the byte field becomes zero.

In [ ]:
def parse_apache_log(line):
    match = APACHE_LOG_PATTERN.match(line)
    if match is None:
        return None

    record = match.groupdict()
    record["status"] = int(record["status"])
    record["bytes"] = 0 if record["bytes"] == "-" else int(record["bytes"])
    record["referrer"] = record["referrer"] or ""
    record["user_agent"] = record["user_agent"] or ""
    return record

## 9. Parse every line with `map`

`map` calls the parser once for every log line. `cache` keeps the parsed results available because several later actions reuse them.

In [ ]:
parsed_or_none_rdd = raw_logs_rdd.map(parse_apache_log).cache()

## 10. Check parsing quality

Do not silently discard malformed input. Count valid and invalid records first, then inspect a few invalid lines.

In [ ]:
valid_count = parsed_or_none_rdd.filter(lambda record: record is not None).count()
invalid_count = parsed_or_none_rdd.filter(lambda record: record is None).count()

print("Valid records  :", valid_count)
print("Invalid records:", invalid_count)

In [ ]:
invalid_lines_rdd = raw_logs_rdd.filter(
    lambda line: parse_apache_log(line) is None
)

invalid_lines_rdd.take(5)

## 11. Keep valid parsed records

From this point, each RDD record is a Python dictionary. No DataFrame is used.

In [ ]:
parsed_logs_rdd = parsed_or_none_rdd.filter(
    lambda record: record is not None
)

parsed_logs_rdd.take(2)

## 12. Count requests by HTTP status

`map` creates `(status, 1)` pairs. `reduceByKey` adds the counts for each status.

In [ ]:
status_counts_rdd = (
    parsed_logs_rdd
    .map(lambda record: (record["status"], 1))
    .reduceByKey(lambda left, right: left + right)
    .sortByKey()
)

status_counts_rdd.collect()

## 13. Find the most requested paths

In [ ]:
path_counts_rdd = (
    parsed_logs_rdd
    .map(lambda record: (record["path"], 1))
    .reduceByKey(lambda left, right: left + right)
)

top_paths = path_counts_rdd.takeOrdered(
    10,
    key=lambda item: (-item[1], item[0]),
)

for path, count in top_paths:
    print(f"{count:>6}  {path}")

## 14. Count requests by method

In [ ]:
method_counts = (
    parsed_logs_rdd
    .map(lambda record: (record["method"], 1))
    .reduceByKey(lambda left, right: left + right)
    .collect()
)

print(sorted(method_counts))

## 15. Calculate total response bytes

`map` selects the byte count and `sum` is the action.

In [ ]:
total_response_bytes = parsed_logs_rdd.map(
    lambda record: record["bytes"]
).sum()

print(f"Total response bytes: {total_response_bytes:,}")

## 16. Convert dictionaries to CSV text

RDDs have no CSV writer. We can still create correctly escaped CSV rows with Python's `csv` module. `mapPartitionsWithIndex` writes the header only in partition 0.

The result remains distributed: HDFS stores a directory containing `_SUCCESS` and one or more `part-*` files.

In [ ]:
import csv
import io

CSV_FIELDS = (
    "client_ip",
    "identity",
    "user",
    "timestamp",
    "method",
    "path",
    "protocol",
    "status",
    "bytes",
    "referrer",
    "user_agent",
)

In [ ]:
def values_to_csv_line(values):
    buffer = io.StringIO()
    writer = csv.writer(buffer, lineterminator="")
    writer.writerow(values)
    return buffer.getvalue()


def partition_to_csv(partition_id, records):
    if partition_id == 0:
        yield values_to_csv_line(CSV_FIELDS)

    for record in records:
        yield values_to_csv_line(record[field] for field in CSV_FIELDS)

In [ ]:
csv_lines_rdd = parsed_logs_rdd.mapPartitionsWithIndex(partition_to_csv)

csv_lines_rdd.take(3)

## 17. Delete the previous output directory

Spark does not overwrite an existing output directory. This command deletes only the D282 CSV output.

In [ ]:
%%bash
HDFS_OUTPUT="/user/$USER/apache-logs/output/csv"

hdfs dfs -rm -r -f "$HDFS_OUTPUT"

## 18. Save the CSV-formatted RDD to HDFS

`saveAsTextFile` is an action and starts a Spark job.

In [ ]:
sc.setJobGroup("save-apache-csv", "Save parsed Apache logs as CSV text")
csv_lines_rdd.saveAsTextFile(hdfs_csv_output)

print("Saved CSV directory:", hdfs_csv_output)

## 19. Verify the HDFS output

The header is the first row of the first part file. Other part files continue with data rows. `getmerge` can combine the ordered part files into one local CSV when a single file is required.

In [ ]:
%%bash
HDFS_OUTPUT="/user/$USER/apache-logs/output/csv"

hdfs dfs -ls -h "$HDFS_OUTPUT"
hdfs dfs -cat "$HDFS_OUTPUT"/part-* | head -5

## 20. Optional: download one merged CSV

This merges the HDFS part files into a local file for inspection. HDFS remains the source of the distributed output.

In [ ]:
%%bash
HDFS_OUTPUT="/user/$USER/apache-logs/output/csv"
LOCAL_CSV='/tmp/d282_apache_logs.csv'

hdfs dfs -getmerge -f "$HDFS_OUTPUT/part-*" "$LOCAL_CSV"
ls -lh "$LOCAL_CSV"
head -5 "$LOCAL_CSV"

## 21. RDD lineage and Spark work

Print the lineage and inspect the Spark UI. `map` and `filter` are narrow transformations. `reduceByKey`, `sortByKey`, and `takeOrdered` may require data movement or multiple stages. An action starts a job, and each stage normally runs one task per partition.

In [ ]:
print(status_counts_rdd.toDebugString().decode("utf-8"))

## 22. Practice

Using only RDD transformations and actions:

1. Find the ten client IP addresses with the most requests.
2. Count successful responses with status codes from 200 through 299.
3. Count client errors from 400 through 499.
4. Find the ten largest responses by byte count.
5. Count requests by day using the date portion of `timestamp`.
6. Save one result as CSV-formatted text in a new HDFS output directory.

In [ ]:
# Write the practice solution here.


## 23. Stop Spark

Run this cell when the lesson is complete.

In [ ]:
parsed_or_none_rdd.unpersist()
spark.stop()
print("Spark session stopped.")